# B-04: Enriched-feature deep-learning rerun of FC-02 (D-41)

Re-runs the FC-02 seq2seq forecasters (DLinear / LSTM / LSTM-VSN) on the **enriched** hourly matrix
`forecast_features_v2.csv` (+7 new future-exog `fx_`: wind-direction sin/cos, is_daytime, 3-sensor SHF,
is_growing/is_winter, days_since_grazing). New `fx_` columns enter the encoder **and** decoder automatically
(channel counts grow via shape inference). Same CV, tracks, models, epochs, GPU as FC-02. Final cell compares
**B04 vs FC-02**.

In [1]:
import sys, time, datetime; sys.path.insert(0, "../../src/models")
from pathlib import Path
import numpy as np, pandas as pd, torch
import forecasting_dl as fdl
HOURLY = Path("../../data/Hourly"); RESULTS = Path("../../results")
dev = fdl.get_device(); print("device:", dev, "| torch", torch.__version__)
m = fdl.load_matrix(HOURLY/"forecast_features_v2.csv"); print("matrix v2", m.shape, "| FX", len(fdl.FX))
W = {"A": fdl.build_windows(m, "A"), "B": fdl.build_windows(m, "B")}
print("enc/dec channels: A enc", W["A"][4]["enc"].shape[-1], "dec", W["A"][4]["dec"].shape[-1],
      "| (FC-02 had enc 23 / dec 19)")

device: cuda | torch 2.11.0+cu128


matrix v2 (210459, 55) | FX 26


enc/dec channels: A enc 30 dec 26 | (FC-02 had enc 23 / dec 19)


## 1  Train + evaluate all models × tracks (30 epochs)

In [2]:
MODELS = ["DLinear", "LSTM", "LSTM_VSN"]; EPOCHS = 30
ALL = []
for mdl in MODELS:
    for trk in ["A", "B"]:
        t0 = time.time()
        rows = fdl.run_track(trk, mdl, W, dev, epochs=EPOCHS)
        ALL += rows
        print(f"{mdl:9s} track {trk}: {len(rows):2d} rows  ({time.time()-t0:.0f}s)", flush=True)
R = pd.DataFrame(ALL); print("total rows", len(R))

DLinear   track A: 15 rows  (11s)


DLinear   track B: 12 rows  (2s)


LSTM      track A: 15 rows  (15s)


LSTM      track B: 12 rows  (3s)


LSTM_VSN  track A: 15 rows  (17s)


LSTM_VSN  track B: 12 rows  (3s)


total rows 81


## 2  Results + save b04_summary.csv

In [3]:
print("=== R2 (towers 4/9) ===")
print(R[R.tower.isin([4,9])].pivot_table(index=["track","tower","model"],columns="horizon",values="R2").round(3).to_string())
print("\n=== Track B (daily) skill vs persistence ===")
for t in [4,9]:
    sub=R[(R.track=="B")&(R.tower==t)]
    print(f"-- Tower {t} --"); print(sub.pivot_table(index="model",columns="horizon",values="skill_persist").round(3).to_string())
R.to_csv(RESULTS/"b04_summary.csv",index=False); print("\nsaved results/b04_summary.csv")

=== R2 (towers 4/9) ===
horizon                  1      3      6      7      12     14     24     48
track tower model                                                           
A     4     DLinear  -0.155    NaN -0.398    NaN -0.396    NaN -0.616 -0.424
            LSTM     -0.188    NaN -0.323    NaN -0.361    NaN -0.446 -0.463
            LSTM_VSN -0.532    NaN -0.097    NaN -0.173    NaN -0.121 -0.107
      9     DLinear  -0.006    NaN -0.158    NaN -0.144    NaN -0.258 -0.193
            LSTM     -0.080    NaN -0.076    NaN -0.226    NaN -0.170 -0.188
            LSTM_VSN -0.262    NaN -0.022    NaN -0.250    NaN -0.269 -0.263
B     4     DLinear   0.337  0.201    NaN  0.163    NaN  0.132    NaN    NaN
            LSTM     -0.292 -0.041    NaN -0.401    NaN -0.680    NaN    NaN
            LSTM_VSN -0.286 -0.192    NaN -0.551    NaN -0.634    NaN    NaN
      9     DLinear   0.292  0.273    NaN  0.242    NaN  0.196    NaN    NaN
            LSTM     -0.072 -0.140    NaN -0.013    

## 3  Compare B04 vs FC-02 (did enriched features help the DL?)

In [4]:
fc02=pd.read_csv(RESULTS/"fc02_summary.csv")
key=["track","tower","horizon","model"]
j=R.merge(fc02[key+["R2","skill_persist","RMSE"]],on=key,suffixes=("_b04","_fc02"))
j["dR2"]=j["R2_b04"]-j["R2_fc02"]; j["dskill"]=j["skill_persist_b04"]-j["skill_persist_fc02"]; j["dRMSE"]=j["RMSE_b04"]-j["RMSE_fc02"]
print("=== mean delta (B04 - FC-02) by track/model, towers 4/9 ===")
print(j[j.tower.isin([4,9])].groupby(["track","model"])[["dR2","dskill","dRMSE"]].mean().round(3).to_string())
print("\n=== best daily R2 per tower — FC-02 vs B04 ===")
for t in [4,9]:
    a=fc02[(fc02.track=="B")&(fc02.tower==t)]["R2"].max(); b=R[(R.track=="B")&(R.tower==t)]["R2"].max()
    print(f"  Tower {t}: FC-02 {a:.3f}  ->  B04 {b:.3f}  (delta {b-a:+.3f})")

=== mean delta (B04 - FC-02) by track/model, towers 4/9 ===
                  dR2  dskill  dRMSE
track model                         
A     DLinear  -0.016  -0.006  0.898
      LSTM      0.025   0.014 -1.638
      LSTM_VSN -0.040  -0.018  3.044
B     DLinear  -0.002  -0.001  0.061
      LSTM      0.119   0.047 -3.383
      LSTM_VSN -0.074  -0.029  2.204

=== best daily R2 per tower — FC-02 vs B04 ===
  Tower 4: FC-02 0.333  ->  B04 0.337  (delta +0.004)
  Tower 9: FC-02 0.326  ->  B04 0.292  (delta -0.034)


## 4  Append to benchmarks.csv (B04)

In [5]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="B04"]
rows=[]
for _,r in R.iterrows():
    rows.append({"replication":"B04","model":r["model"],"tower":f"Tower {int(r.tower)}",
        "feature_set":"enriched_v2 seq2seq (WD/daytime/SHF3/season/days-since-grazing)","track":r["track"],"horizon":int(r["horizon"]),
        "split":"fc_main" if r.tower in (4,9) else "fc_t2folds","R2":r["R2"],"RMSE":r["RMSE"],"MAE":r["MAE"],
        "MBE":r["MBE"],"n_test":int(r["n_test"]),"skill_persist":r["skill_persist"],"skill_clim":r["skill_clim"],
        "WAPE":r["WAPE"],"MASE":r["MASE"],"sMAPE":r["sMAPE"],"MAPE":r["MAPE"],"MAPE_n_excluded":r["MAPE_n_excluded"],
        "date":today,"notes":"B04 enriched-feature rerun of FC-02; native seq2seq; GPU (D-41); WAPE/MASE/sMAPE/MAPE added D-44b"})
new=pd.DataFrame(rows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} B04 rows. Total {len(comb)}.")

Wrote 81 B04 rows. Total 3665.
